# Creating a new tree

Goal: construct a phylogenetic tree of the EspY3 protein family across *Escherichia coli* and relatives (*Shigella* and *Escherichia albertii*. Include in the tree annotated *E. coli* tips with type information.

1. ## BLASTp

### Main BLAST search for tree backbone

RID:HEVX1DZ0014 
Database: NCBI nr
Exluded WP_ to ensure strain-level metadata for *E. coli*
Filters:
- Taxonomy: Bacteria
- Max target sequences: 1000
Download: FASTA sequences > main.faa

This gives a broad *E. coli* representation, and includes Shigella and some unclassified enterobacteriaceae.

### Secondary search for *E. albertii*

A second search was necessary as *E. Albertii* sequences are distant and not represented in the above search, yet the protein appears to share a common ancestor.
This is opposed to doing a secondary search *exluding* *E. coli*. I chose to search specifically for *E. albertii* as this will reduce redundancy in the concatenated files, and resolves the problem of numerous hits of proteins that only share homology across the pentapeptide repeat domain

RID:HEWX62YT016 
Same search parameters except:
- Organism: *Escherichia albertii*
- Max targets: 200
Download: FASTA sequences > albertii_hits.faa

### Processing the data

Now that I have the fasta files, the next steps are to process the data.

1. Concatenate sequences
2. Reduce reduncancy (CD-HIT)
3. Multiple Sequence Allignment + Cleaning

In [7]:
# Concatenate queries

!cat raw/main_944.faa raw/albertii_hits.faa > raw/combined_raw.faa

2. ## CD-HIT to reduce reduncacy

The E. coli hits were all fairly high identity, so a CD-HIT to remove redunant/closely related copies is required.

However, I found in the BLASTp output that there are some *Shigella* hits with very high sequence identity is likely to get lost during the clustering process. Therefore, I will extract any sequences that aren't *E. coli* or *E. albertii* from the combined sequences, run CD-HIT, then rejoin them. 

CD-HIT at 98.5% identity gave 80 clusters which will be ideal for creating a tree once the shigella strains have been added back

In [13]:
# Make a file with everything EXCEPT Shigella (to run CD-HIT on)
!awk '/^>/{p = ($0 ~ /\[Shigella/)} p' raw/combined_raw.faa > raw/extracted_species/shigella.faa

# Make a file with ONLY Shigella (to add back after clustering)
!awk '/^>/{p = !($0 ~ /\[Shigella/)} p' raw/combined_raw.faa > raw/extracted_species/combined_no_shig_raw.faa

In [24]:
!cd-hit \
    -i raw/extracted_species/combined_no_shig_raw.fasta \
    -o cd-hit/combined_cd.fasta \
    -c 0.985

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 21:58:32
Command: cd-hit -i
         raw/extracted_species/combined_no_shig_raw.fasta -o
         cd-hit/combined_cd.fasta -c 0.985

Started: Fri Nov 14 19:25:43 2025
                            Output                              
----------------------------------------------------------------
total seq: 1079
longest and shortest : 543 and 146
Total letters: 552230
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 0M
Buffer          : 1 X 16M = 16M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 82M

Table limit with the given memory limit:
Max number of representatives: 1151167
Max number of word counting entries: 89725337

comparing sequences from          0  to       1079
.
     1079  finished         80  clusters

Approximated maximum memory consumption: 82M
writing new database
writing clustering information
program completed !

Total CPU time 0.09


3. ## Multiple Sequence Alignment and Cleaning

In [ ]:
# Add back the Shigella sequences
!cat cd-hit/combined_cd.fasta raw/extracted_species/shigella.fasta \
      > raw/final_for_alignment.faa

In [4]:
!mafft --maxiterate 1000 --localpair  raw/final_for_alignment.faa > mafft/alligned.faa

outputhat23=16
treein = 0
compacttree = 0
stacksize: 8176 kb
rescale = 1
All-to-all alignment.
tbfast-pair (aa) Version 7.526
alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

outputhat23=16
Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00
tbutree = 1, compacttree = 0
Constructing a UPGMA tree ... 
   90 / 92
done.

Progressive alignment ... 
STEP    66 /91 
Reallocating..done. *alloclen = 2088
STEP    91 /91 
done.
tbfast (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
1 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 16
sueff_global = 0.100000
nadd = 16
Loading 'hat3' ... done.
rescale = 1

   90 / 92
Segment   1/  1    1- 661
STEP 002-018-0  identical.   
Converged.

done
dvtditr (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strate

### Trimal

The allignment is good, but there are a number of empty columns created by insertions present in a few sequences, and in all of the E. albertii strains. 

Therefore, I will use trimal to remove gappy columns, but I will use the original MSA for mapping conservation onto structure.

In [7]:
!trimal -in mafft/alligned.fasta -out trimal/trimal.fasta -automated1

4. ## Tree Construction

Now I will use RAxML-NG to construct a Maximum Likelihood tree and bootstrap values

In [14]:
!raxml-ng --all \
--msa trimal/trimal.fasta \
--model LG+G \
--prefix raxml/newtree \
--bs-trees 100 \
--threads 8 \
--seed 2
    



RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 14-Nov-2025 20:25:17 as follows:

raxml-ng --all --msa trimal/trimal.fasta --model LG+G --prefix raxml/newtree --bs-trees 100 --threads 8 --seed 2

Analysis options:
  run mode: ML tree search + bootstrapping (Felsenstein Bootstrap)
  start tree(s): random (10) + parsimony (10)
  bootstrap replicates: parsimony (100)
  random seed: 2
  tip-inner: OFF
  pattern compression: ON
  per-rate scalers: OFF
  site repeats: ON
  logLH epsilon: general: 10.000000, brlen-triplet: 1000.000000
  fast spr radius: AUTO
  spr subtree cutoff: 1.000000
  fast

5. ## Metadata Retrieval

I want to annotate the tree with E. coli pathotype and serotype information, and annotate the different species to highlight the presence of homologous proteins in Shigella etc.

1. Extract protein IDs from tree
2. Convert protein IDs to Genome IDs (NCBI) (so Enterobase can recognise)
3. 


In [28]:
# Extract each token to get protein IDs
!grep -oE '[A-Z][A-Z0-9_]*\.[0-9]+' make_tree/finaltree.support | sort -u > type_info/protein_ids.txt

In [45]:
from Bio import Entrez, SeqIO
import time
import pandas as pd

Entrez.email = "lachlan.black.2022@uni.strath.ac.uk" 

with open("type_info/protein_ids.txt") as f:
    protein_ids = [line.strip() for line in f if line.strip()]

len(protein_ids), protein_ids[:5]

(92,
 ['AUX62840.1', 'AXO85837.1', 'EEU1436664.1', 'EEV6173070.1', 'EEW3329860.1'])

In [49]:
import re

serotype_pattern = re.compile(r"O\d+(?::H\d+|:NM|:H-)?", re.IGNORECASE)

def infer_serovar_from_text(text: str | None) -> str | None:
    if not isinstance(text, str):
        return None
    m = serotype_pattern.search(text)
    if m:
        return m.group(0)
    return None

In [52]:
def get_metadata_for_protein(pid, sleep=0.3):
    meta = {
        "protein_acc": pid,
        "nucleotide_acc": None,
        "species": None,         
        "strain": None,
        "host": None,
        "isolation_source": None,
        "country": None,
        "collection_date": None,
        "serovar": None,
        "biosample": None,
    }
    
    try:
        # 1) Fetch protein record
        handle = Entrez.efetch(db="protein", id=pid, rettype="gb", retmode="text")
        record = SeqIO.read(handle, "genbank")
        handle.close()
        
        organism = record.annotations.get("organism", "")
        meta["species"] = organism   # usually "Escherichia coli" or "Escherichia albertii"
        description = record.description
        
        protein_source_feat = None
        cds_feat = None
        
        for feat in record.features:
            if feat.type == "source":
                protein_source_feat = feat
            elif feat.type == "CDS":
                cds_feat = feat
        
        # 2) Protein source qualifiers
        if protein_source_feat is not None:
            q = protein_source_feat.qualifiers
            
            if "strain" in q:
                meta["strain"] = q["strain"][0]
            if "host" in q:
                meta["host"] = q["host"][0]
            if "isolation_source" in q:
                meta["isolation_source"] = q["isolation_source"][0]
            if "country" in q:
                meta["country"] = q["country"][0]
            if "collection_date" in q:
                meta["collection_date"] = q["collection_date"][0]
            
            for xref in q.get("db_xref", []):
                if xref.startswith("BioSample:"):
                    meta["biosample"] = xref.split(":", 1)[1]
            
            sero_candidates = []
            for key in ["serovar", "serotype", "serogroup", "serospecies"]:
                if key in q:
                    sero_candidates.append(q[key][0])
            if sero_candidates:
                meta["serovar"] = "; ".join(sero_candidates)
        
        # 3) coded_by → nucleotide
        nuc_acc = None
        if cds_feat is not None:
            coded_by = cds_feat.qualifiers.get("coded_by", [""])[0] if "coded_by" in cds_feat.qualifiers else ""
            if coded_by:
                inside = coded_by
                if "(" in coded_by and ")" in coded_by:
                    inside = coded_by.split("(", 1)[1].split(")", 1)[0]
                nuc_acc = inside.split(":", 1)[0]
                meta["nucleotide_acc"] = nuc_acc
        
        # 4) Fill missing bits from nucleotide record if needed
        need_nuc = (
            nuc_acc
            and (
                meta["strain"] is None
                or meta["host"] is None
                or meta["isolation_source"] is None
                or meta["country"] is None
                or meta["collection_date"] is None
                or meta["serovar"] is None
                or meta["biosample"] is None
            )
        )
        
        if need_nuc:
            try:
                h = Entrez.efetch(db="nuccore", id=nuc_acc, rettype="gb", retmode="text")
                n_record = SeqIO.read(h, "genbank")
                h.close()
                
                for feat in n_record.features:
                    if feat.type == "source":
                        q = feat.qualifiers
                        
                        if meta["strain"] is None and "strain" in q:
                            meta["strain"] = q["strain"][0]
                        if meta["host"] is None and "host" in q:
                            meta["host"] = q["host"][0]
                        if meta["isolation_source"] is None and "isolation_source" in q:
                            meta["isolation_source"] = q["isolation_source"][0]
                        if meta["country"] is None and "country" in q:
                            meta["country"] = q["country"][0]
                        if meta["collection_date"] is None and "collection_date" in q:
                            meta["collection_date"] = q["collection_date"][0]
                        
                        if meta["biosample"] is None:
                            for xref in q.get("db_xref", []):
                                if xref.startswith("BioSample:"):
                                    meta["biosample"] = xref.split(":", 1)[1]
                        
                        if meta["serovar"] is None:
                            sero_candidates = []
                            for key in ["serovar", "serotype", "serogroup", "serospecies"]:
                                if key in q:
                                    sero_candidates.append(q[key][0])
                            if sero_candidates:
                                meta["serovar"] = "; ".join(sero_candidates)
                        
                        break
            except Exception as e2:
                print(f"(Warning) nucleotide fetch failed for {pid} / {nuc_acc}: {e2}")
        
        # 5) Optional: infer serovar from text if still missing
        if meta["serovar"] is None:
            text_blob = " | ".join(
                t for t in [description, organism, meta["strain"], meta["isolation_source"]] if isinstance(t, str)
            )
            inferred = infer_serovar_from_text(text_blob)
            if inferred:
                meta["serovar"] = inferred
        
        time.sleep(sleep)
        return meta
    
    except Exception as e:
        print(f"ERROR for {pid}: {e}")
        time.sleep(sleep)
        return meta

In [53]:
rows = []

for i, pid in enumerate(protein_ids, start=1):
    m = get_metadata_for_protein(pid)
    rows.append(m)
    if i % 10 == 0:
        print(f"Processed {i}/{len(protein_ids)} proteins...")

meta_df = pd.DataFrame(rows)

# Column order including species
meta_df = meta_df[
    [
        "protein_acc",
        "nucleotide_acc",
        "species",          # <--- here
        "strain",
        "host",
        "isolation_source",
        "country",
        "collection_date",
        "serovar",
        "biosample",
    ]
]

meta_df.to_csv("type_info/protein_metadata.csv", index=False)
meta_df.head()

Processed 10/92 proteins...
Processed 20/92 proteins...
Processed 30/92 proteins...
Processed 40/92 proteins...
Processed 50/92 proteins...
Processed 60/92 proteins...
Processed 70/92 proteins...
Processed 80/92 proteins...
Processed 90/92 proteins...


,protein_acc,nucleotide_acc,species,strain,host,isolation_source,country,collection_date,serovar,biosample
0,AUX62840.1,CP021726.1,Escherichia coli,Combat11I9,Homo sapiens,urine,None,Oct-2004,None,None
1,AXO85837.1,CP031609.1,Escherichia coli,N3,Homo sapiens,Rectal swab,None,2015,None,None
2,EEU1436664.1,AARIEB010000010.1,Escherichia coli,712542,Homo sapiens,None,None,Mar-2019,Escherichia coli,None
3,EEV6173070.1,AARMPV010000001.1,Escherichia coli,FSIS11814928,None,animal-cattle-dairy cow,None,2018,None,None
4,EEW3329860.1,AAROSG010000020.1,Escherichia albertii,PNUSAE019256,None,None,None,None,None,None


6. ## Tree annotation

With the protein metadata, I manually organised the isolation source into the following categories:
    1. human (any source coming from a human, i.e. fecal, blood, etc.)
    2. animal (any source from an animal, i.e. manure, bird feces etc. NOTE: if the protein was isolated from a food source like chicken, this has been grouped with food)
    3. food
    4. environmental (i.e. waste water...)
    5. unknown

The below script converts the resulting csv into a format that can be uploaded to the iTOL tree
    

In [6]:
import pandas as pd

# === 1. Load your metadata CSV ===
# Change this filename if needed
df = pd.read_csv("protein_tree/type_info/proteinid_source.csv")   # must contain columns: protein_acc, source

# === 2. Define your 5 isolation sources and colours ===
category_order = ["human", "animal", "food", "environmental", "unknown"]

color_map = {
    "human": "#e41a1c",          # red
    "animal": "#377eb8",         # blue
    "food": "#4daf4a",           # green
    "environmental": "#984ea3",  # purple
    "unknown": "#ff7f00",        # orange
}

label_map = {
    "human": "Human",
    "animal": "Animal",
    "food": "Food",
    "environmental": "Environmental",
    "unknown": "Unknown",
}

# === 3. Build the iTOL COLORSTRIP dataset ===
output = []
output.append("DATASET_COLORSTRIP")
output.append("SEPARATOR TAB")
output.append("DATASET_LABEL\tIsolation Source")
output.append("COLOR\t#000000\n")  # default dataset colour (not very important)

# Legend (fixed order)
legend_shapes = "\t".join(["1"] * len(category_order))
legend_colors = "\t".join(color_map[c] for c in category_order)
legend_labels = "\t".join(label_map[c] for c in category_order)

output.append("LEGEND_TITLE\tIsolation Source")
output.append(f"LEGEND_SHAPES\t{legend_shapes}")
output.append(f"LEGEND_COLORS\t{legend_colors}")
output.append(f"LEGEND_LABELS\t{legend_labels}")
output.append("\nDATA")

# === 4. Add one line per protein ===
# Format: <leaf_id> <color> <optional_label>
for _, row in df.iterrows():
    raw_source = str(row["source"])
    src_norm = raw_source.strip().lower()

    if src_norm not in category_order:
        print(f"Warning: unknown source '{raw_source}' for protein {row['protein_acc']}. Treating as 'unknown'.")
        src_norm = "unknown"

    color = color_map[src_norm]
    label = label_map[src_norm]
    protein_id = row["protein_acc"]

    output.append(f"{protein_id}\t{color}\t{label}")

# === 5. Write to file ===
with open("protein_tree/type_info/itol_isolation_colorstrip.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(output))

print("Generated itol_isolation_colorstrip.txt")

Generated itol_isolation_colorstrip.txt
